In [1]:
import generate_data

Данные для работы успешно сгенерированны! Размер строки в файлах - 200 байт


---

In [2]:
from generate_data import generate_vin, generate_model_name, generate_brand, generate_date, get_status

In [3]:
from datetime import datetime, timedelta
from collections import defaultdict
import random, decimal
import pandas as pd
import bisect
import os

## Импорт классов и разработка методов приложения:

**`Импорт классов:`**

In [4]:
from models import Car, CarFullInfo, CarStatus, Model, ModelSaleStats, Sale

---

**`Тест импортированных классов:`**

**Объект CarStatus** (Статус доступности автомобиля):

In [5]:
test_status = CarStatus(get_status())

In [6]:
test_status

<CarStatus.reserve: 'reserve'>

**Объект Model** (Модель автомобиля):

In [7]:
test_model = Model(
    id = random.randint(501,1001),
    name = generate_model_name(),
    brand = generate_brand())

In [8]:
test_model

Model(id=668, name='E-3', brand='U')

**Объект Car** (Автомобиль):

In [9]:
test_car = Car(
    vin = generate_vin(), 
    model = random.randint(1,501), 
    price = decimal.Decimal(random.randint(2000,4000)),
    date_start = generate_date(2020, 2024),
    status = CarStatus(get_status())
)

In [10]:
test_car

Car(vin='0RFEQ5X54JB660819', model=112, price=Decimal('2167'), date_start=datetime.datetime(2020, 7, 24, 0, 0), status=<CarStatus.available: 'available'>)

**Объект Sale** (Продажа):

In [12]:
test_sale = Sale(
    sales_number = f'{random.randint(110,130)}#{random.randint(1, 1000)}',
    car_vin = '3WVBW2M56ID967749',#generate_vin(),
    sales_date = generate_date(2020, 2024),
    cost = round(decimal.Decimal(random.randint(3000,4000)+random.random()),2)
)

In [13]:
test_sale

Sale(sales_number='119#892', car_vin='3WVBW2M56ID967749', sales_date=datetime.datetime(2023, 7, 12, 0, 0), cost=Decimal('3547.63'))

---

Разработка методов класса:

##### **Методы по отдельности**:

###### 1.1 Добавить автомобиль:

In [14]:
class CarService:
    
    def __init__(self, root_directory_path: str) -> None:
        self.root_directory_path = root_directory_path
        self.FIXED_LINE_SIZE = 200
       
    def add_car(self, car: Car) -> Car:
        ''' Метод добавляет строку об автомобиле в файл cars.txt
            ВАЖНО: 
                1. Файл cars.txt невозможно полностью загрузить в ОЗУ
                2. Файл индекса cars_index.txt можно загрузить в ОЗУ
        '''
        car.date_start = datetime.date(car.date_start)
        new_vin = car.vin
        new_model = car.model
        
        # Проверка наличия переданного id модели в справочнике 
        # (Проверка валидности значения для поля foreign key)
        with open(f'{self.root_directory_path}models_index.txt', mode='r', encoding='utf-8') as models_f:
            # Обработка строки заголовка для динамического обращения к полю
            models_header = models_f.readline().strip().split(';')
            id_pos = models_header.index('key')
            lines = models_f.readlines()
            
            id_values = [int(line.strip().split(';')[id_pos]) for line in lines]

        if car.model not in id_values:
            raise ValueError('''Введённого ID модели не существует в базе; 
            Исправьте значение или обновите справочник моделей.
            ''')

        # Поиск значения индекса для нового автомобиля 
        with open(f'{self.root_directory_path}cars_index.txt', 'r', encoding='utf-8') as cars_index_f:
            cars_index_header = cars_index_f.readline().strip().split(';')
            car_key_pos = cars_index_header.index('key')
            car_index_pos = cars_index_header.index('index')
            # Сохраняем в список строки индекса
            car_index_lines = cars_index_f.readlines()
            
        # Множество VIN для последующей проверки нового VIN + новый индекс
        car_vin_set = set([line.strip().split(';')[car_key_pos] for line in car_index_lines])
        index_for_new_car = max( set( [int(line.strip().split(';')[car_index_pos]) for line in car_index_lines] ) ) + 1
            
        # Проверка VIN на наличие в базе
        if new_vin in car_vin_set:
            raise ValueError('''Автомобиль с данным VIN уже существует в базе;
            Убедитесь в правильности значения.''')
        else: 
            # Запись строки с новым автомобилем в конец файла 
            with open(f'{self.root_directory_path}cars.txt', mode='a', encoding='utf-8', newline='') as cars_f:
                new_cars_line = ';'.join([str(a[-1]) for a in car]).ljust(self.FIXED_LINE_SIZE-1)+'\n' # a - кортеж вида (Атрибут, Значение)
                cars_f.write(new_cars_line)
                
            # Обновление индекса
            car_index_new_line = (new_vin + ';' + str(index_for_new_car)).ljust(self.FIXED_LINE_SIZE-1) + '\n'
            car_index_lines.append(car_index_new_line)
            car_index_lines_sorted = sorted(car_index_lines) # Так как VIN находится в начале строки, применяем самую обычную сортировку
            
            # Промежуточный файл
            with open(f'{self.root_directory_path}temp_cars_index.txt', 'w', encoding='utf-8', newline='') as temp_f:
                cars_index_header_rebuilt = ';'.join(cars_index_header).ljust(self.FIXED_LINE_SIZE-1)+'\n'
                temp_f.write(cars_index_header_rebuilt)
                for line in car_index_lines_sorted:
                    temp_f.write(line)
                    
            new_cars_index_file = f'{self.root_directory_path}temp_cars_index.txt'
            old_cars_index_file = f'{self.root_directory_path}cars_index.txt'
            # Финальная перезапись индекса 
            os.replace(new_cars_index_file, old_cars_index_file)
        
        return car        
        # Бинарный поиск места для вставки за O(log N)
        #insert_place = bisect.bisect_left(lines, new_index_line)
        #print(new_index_line)
        #print(insert_place)
        #lines.insert(insert_place, new_index_line)

In [15]:
CarService('data/').add_car(
    Car(
        vin = generate_vin(), 
        model = random.randint(1,501), 
        price = decimal.Decimal(random.randint(2000,4000)),
        date_start = generate_date(2020, 2024),
        status = CarStatus('available') #get_status())
    )
)

Car(vin='0NTUF2I09UB681780', model=269, price=Decimal('3294'), date_start=datetime.date(2022, 8, 8), status=<CarStatus.available: 'available'>)

###### 1.2 Добавить модель в справочник

In [16]:
class CarService:
    
    def __init__(self, root_directory_path: str) -> None:
        self.root_directory_path = root_directory_path
        self.FIXED_LINE_SIZE = 200

    def __get_min_free_model_id(self):
        ''' Метод возвращает минимальный id таблицы Models, доступный для записи '''
        with open(f'{self.root_directory_path}models_index.txt', mode='r', encoding='utf-8') as models_index_f:
            # Обработка заголовка
            models_index_header = models_index_f.readline().strip().split(';')
            models_index_lines = models_index_f.readlines()
            models_key_pos = models_index_header.index('key')
            i = 1
            ids = set([int(line.strip().split(';')[models_key_pos]) for line in models_index_lines])
            while i in ids:
                i += 1
            return i

    def add_model(self, model: Model):
        ''' Метод добавляет новую запись в таблицу-справочник Models, перезаписывая файл '''
        # 1. Чтение файла с индексом моделей 
        with open(f'{self.root_directory_path}models_index.txt', mode='r', encoding='utf-8') as models_index_f:
            models_index_header = models_index_f.readline().strip().split(';')
            model_id_pos = models_index_header.index('key')
            model_index_pos = models_index_header.index('index') 
            # Список необработанных строк
            models_index_lines = models_index_f.readlines()

        model_id_set = set( [ int(line.strip().split(';')[model_id_pos]) for line in models_index_lines ]  )
        new_models_index = max( [ int(line.strip().split(';')[model_index_pos]) for line in models_index_lines ] ) + 1

        if model.id in model_id_set:
            model.id = self.__get_min_free_model_id()

        # Обновление индекса
        with open(f'{self.root_directory_path}models_index.txt', mode='w+', encoding='utf-8', newline='') as models_index_f:
            new_index_line_parts = [None, None]
            new_index_line_parts[model_id_pos] = str(model.id)
            new_index_line_parts[model_index_pos] = str(new_models_index) 
            
            new_index_line = ';'.join(new_index_line_parts).ljust(self.FIXED_LINE_SIZE-1) + '\n'
            
            models_index_lines.append(new_index_line)            
            models_index_parsed_lines = [line.strip().split(';') for line in models_index_lines]
            sorted_lines = sorted(models_index_parsed_lines, key=lambda x: int(x[model_id_pos]))

            # Перезапись файла
            models_index_f.write(';'.join(models_index_header).ljust(self.FIXED_LINE_SIZE-1) + '\n')
            for line in sorted_lines:
                models_index_f.write( ';'.join(line).ljust(self.FIXED_LINE_SIZE-1) + '\n' )
            
        print(new_index_line)
        
        # Добавление строки в файл 
        with open(f'{self.root_directory_path}models.txt', mode='a+', encoding='utf-8', newline='') as models_f:
            # Перемещение в начало файла для обработки заголовка
            models_f.seek(0)
            models_header = models_f.readline().strip().split(';')
            model_id_pos = models_header.index('id')
            model_name_pos = models_header.index('name')
            model_brand_pos = models_header.index('brand')
            
            # Перемещение в конец файла
            models_f.seek(0,2)

            # Реконструкция строки
            new_model_line_part = [None, None, None]
            new_model_line_part[model_name_pos] = model.name
            new_model_line_part[model_id_pos] = str(model.id)
            new_model_line_part[model_brand_pos] = model.brand
            # Преобразование строки
            new_model_line = ';'.join(new_model_line_part).ljust(self.FIXED_LINE_SIZE-1)+'\n'
            # Запись
            models_f.write(new_model_line)

        return model

In [17]:
CarService('data/').add_model(test_model)

668;479                                                                                                                                                                                                



Model(id=668, name='E-3', brand='U')

###### 2. Добавить продажу

In [18]:
class CarService:
    def __init__(self, root_directory_path: str) -> None:
        self.root_directory_path = root_directory_path
        self.FIXED_LINE_SIZE = 200
        
   # Задание 2. Сохранение продаж.
    def sell_car(self, sale: Sale) -> Car:
        #pass
        car_vin = sale.car_vin
        sales_number = sale.sales_number
        cost = str(sale.cost)
        sales_date = str(datetime.date(sale.sales_date))
        new_car_status = 'sold'
        
        #1. Загружаем индекс в словарь { 'VIN': номер_строки }
        car_index = {}
        with open(f'{self.root_directory_path}cars_index.txt', 'r', encoding='utf-8') as cars_index_f:
            cars_index_header = cars_index_f.readline().strip().split(';')
            cars_key_pos = cars_index_header.index('key')
            cars_index_pos = cars_index_header.index('index')
            
            for line in cars_index_f:
                vin = line.strip().split(';')[cars_key_pos]
                cars_index_num = line.strip().split(';')[cars_index_pos]
                car_index[vin] = int(cars_index_num)
                    
        if car_vin not in car_index:
            raise ValueError(f"VIN-номер не найден в индексе; Убедитесь в правильности значения.")

        # Находим переданный vin и его номер строки
        line_number = car_index[car_vin]
    
        #2. ВЫЧИСЛЯЕМ СМЕЩЕНИЕ
        # Прибавляем 1, пропусукая заголовок
        jump_size = (line_number) * self.FIXED_LINE_SIZE
    
        # 3. Обновляем статус в cars.txt
        with open(f'{self.root_directory_path}cars.txt', 'r+b') as car_file:
            # Обработка заголовка
            cars_header = car_file.readline().decode('utf-8').strip().split(';')
            status_pos = cars_header.index('status')
            # Переход на строку
            car_file.seek(jump_size)
            # Декодируем и убираем пробелы справа у найденной строки
            decoded_line = car_file.readline().decode('utf-8')
            parts = decoded_line.strip().split(';')
            # Меняем статус
            if parts[status_pos] == 'sold':
                raise ValueError('Введенному VIN-номеру уже соответствует статус "sold"')
            parts[status_pos] = new_car_status
            
            # Собираем обратно и дополняем пробелами до нужного размера
            updated_line_str = ';'.join(parts)
            final_line_str = updated_line_str.ljust(self.FIXED_LINE_SIZE- 1) + '\n'
            final_bytes = final_line_str.encode('utf-8')
            
            # Возвращение обратно в начало строки + перезапись
            car_file.seek(jump_size)
            car_file.write(final_bytes)
    
        # 4. Записываем продажу в sales.txt
        with open(f'{self.root_directory_path}sales.txt', 'a', encoding='utf-8', newline='') as sales_file:
            sales_file.write(f'{sales_number};{car_vin};{cost};{sales_date}'.ljust(self.FIXED_LINE_SIZE-1)+'\n')
            
        # 5. Добавляем строку в sales_index.txt 
        with open(f'{self.root_directory_path}sales_index.txt', 'r+', newline='') as sales_index_f:
            # Обработка заголовка для динамического обращения к элементам строки
            sales_index_header = sales_index_f.readline().strip().split(';')
            sales_index_pos = sales_index_header.index('index')
            sales_key_pos = sales_index_header.index('key')
            
            sales_index_parsed_lines = []
            for line in sales_index_f:
                sales_index_parsed_lines.append(line.strip().split(';'))
            # Вычисление значения индекса для новой строки
            new_index_num = max( set( [int(line[sales_index_pos]) for line in sales_index_parsed_lines] ) ) + 1
            new_line_parts = [None, None]
            new_line_parts[sales_key_pos] = car_vin
            new_line_parts[sales_index_pos] = str(new_index_num)
            # Сортировка с учётом новой строки
            sales_index_parsed_lines.append(new_line_parts)
            lines_to_write = sorted(sales_index_parsed_lines, key=lambda x: x[sales_key_pos])
            # Перезапись
            sales_index_f.seek(0)
            sales_index_f.truncate()
            sales_index_f.write(';'.join(sales_index_header).ljust(self.FIXED_LINE_SIZE-1)+'\n')
            
            for line in lines_to_write:
                joined_line = ';'.join(line).ljust(self.FIXED_LINE_SIZE-1)+'\n'
                sales_index_f.write(joined_line) 
            
        return sale

In [19]:
CarService('data/').sell_car(test_sale)

Sale(sales_number='119#892', car_vin='3WVBW2M56ID967749', sales_date=datetime.datetime(2023, 7, 12, 0, 0), cost=Decimal('3547.63'))

###### 3. Вывод доступных к продаже авто:

In [20]:
class CarService:
    def __init__(self, root_directory_path: str) -> None:
        self.root_directory_path = root_directory_path
        self.FIXED_LINE_SIZE = 200

    def get_cars(self, status: CarStatus) -> list[Car]:
        
        total_list = []
        
        with open(f'{self.root_directory_path}cars.txt', mode='r', encoding='utf-8') as cars_f:
            # Обращение к заголовку
            cars_header = cars_f.readline().strip().split(';')
            status_pos = cars_header.index('status')
            vin_pos = cars_header.index('vin')
            model_id_pos = cars_header.index('model')
            price_pos = cars_header.index('price')
            date_start_pos = cars_header.index('date_start')
            # Построчное чтение строк после заголовка
            for line in cars_f:
                parsed_line = line.strip().split(';')
                if parsed_line[status_pos] == str(status):
                    total_list.append(parsed_line)

        result_list = []

        # Преобразование с учётом сортировки
        for parsed_line in sorted(total_list, key = lambda x: x[vin_pos]):
            result_list.append(
                Car(
                    vin=parsed_line[vin_pos],
                    model=int(parsed_line[model_id_pos]),
                    price=decimal.Decimal(parsed_line[price_pos]),
                    date_start=datetime.strptime(parsed_line[date_start_pos], '%Y-%m-%d'),
                    status=CarStatus(parsed_line[status_pos])
                )
            )

        return result_list

        ### РЕШЕНИЕ С ПОМОЩЬЮ PANDAS: (Без учёта пробелов справа)
        #columns=['vin','model','price','date_start','status']
        #chunks_list = []
        #print(str(status))

        ## Проход по данным по частям (
        #for chunk in pd.read_csv(f'{self.root_directory_path}cars.txt', sep=';', chunksize=100_000):
         #   chunks_list.append(chunk[chunk['status'] == str(status)])

        ## Конкатенация найденных проданных автомобилей
        #total_df = pd.concat(chunks_list)
        ## Список для вывода результата
        #total_list = []

        ## Превращаем строки таблицы в кортежи
        #for row in total_df.itertuples(index=False):
         #   total_list.append(
          #      Car(
           #         vin=row.vin,
            #        model=row.model,
             #       price=row.price,
              #      date_start=row.date_start,
               #     status=CarStatus(row.status)
                #)
            #)  
        
        #return total_list

In [21]:
CarService('data/').get_cars('available')

[Car(vin='0AAAA6R32VH044942', model=90, price=Decimal('3986.5'), date_start=datetime.datetime(2024, 7, 17, 0, 0), status=<CarStatus.available: 'available'>),
 Car(vin='0AAAC5Z62TD677833', model=7, price=Decimal('3467.44'), date_start=datetime.datetime(2022, 3, 16, 0, 0), status=<CarStatus.available: 'available'>),
 Car(vin='0AABV9U93LC365899', model=45, price=Decimal('3793.18'), date_start=datetime.datetime(2023, 11, 4, 0, 0), status=<CarStatus.available: 'available'>),
 Car(vin='0AACQ8S63LN121528', model=2, price=Decimal('2947.26'), date_start=datetime.datetime(2020, 10, 29, 0, 0), status=<CarStatus.available: 'available'>),
 Car(vin='0AAFF5X02MN854294', model=213, price=Decimal('2922.52'), date_start=datetime.datetime(2023, 6, 26, 0, 0), status=<CarStatus.available: 'available'>),
 Car(vin='0AAHR0L55UC004146', model=192, price=Decimal('2034.26'), date_start=datetime.datetime(2022, 11, 3, 0, 0), status=<CarStatus.available: 'available'>),
 Car(vin='0AAHY2Z32KK078627', model=310, price

###### 4. Детальная информация по VIN:

In [22]:
class CarService:
    def __init__(self, root_directory_path: str) -> None:
        self.root_directory_path = root_directory_path
        self.FIXED_LINE_SIZE = 200
    
    def get_car_info(self, vin: str) -> CarFullInfo | None:
        result_dict = {}
        result_dict['vin'] = vin
        FIXED_LINE_SIZE = 200
        # Обращение к индексу
        with open(f'{self.root_directory_path}cars_index.txt', mode='r') as index_f:
            # Обработка заголовка для динамического обращения к элементам строки
            index_header = index_f.readline().strip().split(';')
            key_position = index_header.index('key')
            index_position = index_header.index('index')
            
            required_line = []
            # Перебор строк
            for i,row in enumerate(index_f):
                if row.strip().split(';')[0] == vin:
                    # Сохранение индекса
                    required_line = row.strip().split(';') 
                    old_vin_index = required_line[index_position]
                    # Сохранение номера с учётом пропуска заголовка
                    row_number = i+1             
                    break # Отбрасываем перебор после найденного VIN
                    
            if required_line == []: # Предупреждение об отсутствии VIN
                raise ValueError(f'VIN не найден, убедитесь в правильности написания')

        # Открытие файла cars.txt с возможность вставки значений
        with open(f'{self.root_directory_path}cars.txt', mode='r') as cars_f:
            # Обработка заголовка для динамического обращения к элементам строки
            cars_header = cars_f.readline().strip().split(';')
            vin_position = cars_header.index('vin')
            model_position = cars_header.index('model')
            price_position = cars_header.index('price')
            date_position = cars_header.index('date_start')
            status_position = cars_header.index('status')
            
            # Поиск строки по индексу (перемещение курсора)
            cars_f.seek(int(old_vin_index)*FIXED_LINE_SIZE)
            # Преобразование строки в список значений
            required_car = cars_f.readline().strip().split(';')

        result_dict['model_id'] = required_car[model_position]
        result_dict['date_start'] = required_car[date_position]
        result_dict['price'] = required_car[price_position]
        result_dict['status'] = required_car[status_position]

        with open(f'{self.root_directory_path}models_index.txt', mode='r') as models_index_f:
            # Обработка заголовка для динамического обращения к элементам строки
            index_header = models_index_f.readline().strip().split(';')
            key_position = index_header.index('key')
            index_position = index_header.index('index')
            
            required_model_line = []
            # Перебор строк
            for i,row in enumerate(models_index_f):
                if row.strip().split(';')[key_position] == result_dict['model_id']:
                    # Сохранение индекса
                    required_model_line = row.strip().split(';') 
                    model_id_index = required_model_line[index_position]
                    # Сохранение номера с учётом пропуска заголовка
                    model_row_number = i+1             
                    break # Отбрасываем перебор после найденного VIN
                    
            if required_model_line == []: # Предупреждение об отсутствии VIN
                raise ValueError(f'ID модели не найден, убедитесь в корректности данных')

        with open(f'{self.root_directory_path}models.txt', mode='r') as models_f:
            # Обработка заголовка для динамического обращения к элементам строки
            models_header = models_f.readline().strip().split(';')
            name_position = models_header.index('name')
            brand_position = models_header.index('brand')

            models_f.seek(int(model_id_index)*FIXED_LINE_SIZE)
            model_row = models_f.readline().strip().split(';')

        result_dict['model_name'] = model_row[name_position]
        result_dict['brand'] = model_row[brand_position]
        
        if result_dict['status'] == 'sold':
            with open(f'{self.root_directory_path}sales.txt') as sales_f, open(f'{self.root_directory_path}sales_index.txt') as sales_idx_f:
                # ОБРАЩЕНИЕ К ИНДЕКСУ
                sales_idx_header = sales_idx_f.readline().strip().split(';')
                sales_key_pos = sales_idx_header.index('key')
                sales_idx_pos = sales_idx_header.index('index')
                sales_row_number = False
                # Перебор строк
                for i,row in enumerate(sales_idx_f):
                    if row.strip().split(';')[sales_key_pos] == result_dict['vin']:
                        # Сохранение индекса
                        required_sales_line = row.strip().split(';') 
                        sales_index = int(required_sales_line[sales_idx_pos])
                        # Сохранение номера с учётом пропуска заголовка
                        sales_row_number = i+1             
                        break # Отбрасываем перебор после найденного VIN
                # Если номер продажи успешно найден
                if sales_row_number != False:
                    # ОБРАЩЕНИЕ К ФАЙЛУ ПРОДАЖ
                    sales_header = sales_f.readline().strip().split(';')
                    print(sales_header)
                    sales_date_position = sales_header.index('sales_date')
                    sales_cost_position = sales_header.index('cost')

                    sales_f.seek(int(sales_index)*FIXED_LINE_SIZE)
                    sales_row = sales_f.readline().strip().split(';')
                    result_dict['sales_date'] = sales_row[sales_date_position]
                    result_dict['sales_cost'] = sales_row[sales_cost_position]
                else:
                    result_dict['sales_date'] = None
                    result_dict['sales_cost'] = None
                    # или raise ValueError('НАЙДЕНА ОШИБКА: ОТСУТСТВУЕТ ПРОДАЖА В ТАБЛИЦЕ SALES')
        else: 
            # В случае, когда статус не равен "sold"
            result_dict['sales_date'] = None
            result_dict['sales_cost'] = None

        #print(result_dict)
        
        result_info = CarFullInfo(
            vin = result_dict['vin'],
            date_start = datetime.strptime(result_dict['date_start'], '%Y-%m-%d'),
            price = decimal.Decimal(result_dict['price']),
            status = CarStatus(result_dict['status']),
            car_model_name = result_dict['model_name'],
            car_model_brand = result_dict['brand'],
            sales_date = datetime.strptime(result_dict['sales_date'], '%Y-%m-%d'),
            sales_cost = decimal.Decimal(result_dict['sales_cost'])
        )
        
        return result_info
        

In [23]:
CarService('data/').get_car_info('3WVBW2M56ID967749')

['sales_number', 'car_vin', 'cost', 'sales_date']


CarFullInfo(vin='3WVBW2M56ID967749', car_model_name='H-8', car_model_brand='V', price=Decimal('2856.63'), date_start=datetime.datetime(2024, 2, 16, 0, 0), status=<CarStatus.sold: 'sold'>, sales_date=datetime.datetime(2023, 7, 12, 0, 0), sales_cost=Decimal('3547.63'))

###### 5. Редактирование VIN:

In [24]:
class CarService:
    def __init__(self, root_directory_path: str) -> None:
        self.root_directory_path = root_directory_path
        self.FIXED_LINE_SIZE = 200

    def update_vin(self, vin: str, new_vin: str) -> Car:
        # Обращение к индексу
        with open(f'{self.root_directory_path}cars_index.txt', mode='r') as index_f:
            # Обработка заголовка для динамического обращения к элементам строки
            cars_index_header = index_f.readline().strip().split(';')
            key_position = cars_index_header.index('key')
            index_position = cars_index_header.index('index')
            
            required_line = []
            # Перебор строк
            for i,row in enumerate(index_f):
                if row.strip().split(';')[key_position] == vin:
                    # Сохранение индекса
                    required_line = row.strip().split(';') 
                    old_vin_index = required_line[index_position]
                    # Сохранение номера строки с учётом пропуска заголовка для возвращения в процессе изменения
                    row_number = i+1             
                    break # Отбрасываем перебор после найденного VIN
                    
            if required_line == []: # Предупреждение об отсутствии VIN
                raise ValueError(f'VIN не найден, убедитесь в правильности написания')

        # Открытие файла cars.txt с возможность вставки значений
        with open(f'{self.root_directory_path}cars.txt', mode='r+b') as cars_f:
            # Обработка заголовка для динамического обращения к элементам строки
            cars_header = cars_f.readline().decode('utf-8').strip().split(';') # Список
            vin_position = cars_header.index('vin')
            model_position = cars_header.index('model')
            price_position = cars_header.index('price')
            date_position = cars_header.index('date_start')
            status_position = cars_header.index('status')
            
            # Поиск строки по индексу (перемещение курсора)
            cars_f.seek(int(old_vin_index)*self.FIXED_LINE_SIZE)
            # Преобразование строки в список значений
            required_car = cars_f.readline().decode('utf-8').strip().split(';')
            # !!! Редактирование значения !!!
            required_car[vin_position] = new_vin
            # Преобразование в строку
            changed_required_car = (';'.join(required_car).ljust(self.FIXED_LINE_SIZE-1)+'\n').encode('utf-8')
            # Возвращение курсора в начало строки
            cars_f.seek(int(old_vin_index)*self.FIXED_LINE_SIZE)
            # !!! Изменение строки !!!
            cars_f.write(changed_required_car)

        # Обновление индекса
        with open(f'{self.root_directory_path}cars_index.txt', mode='r+b') as index_f, \
             open(f'{self.root_directory_path}sorted_cars_index.txt', mode='w', newline='') as new_index_f:
            # ----!!! Изменение VIN в найденной строке индекса !!!----
            required_line[key_position] = new_vin
            # Преобразование строки для записи
            changed_idx_line = (';'.join(required_line).ljust(self.FIXED_LINE_SIZE-1)+'\n').encode('utf-8')
            # Перемещаем курсор 
            index_f.seek(row_number*self.FIXED_LINE_SIZE)
            # ----!!! Изменение строки !!!----
            index_f.write(changed_idx_line)

            # Сортировка строк
            index_f.seek(0) # Перемещение в начало файла  
            index_f.seek(self.FIXED_LINE_SIZE)  # Пропускаем заголовок
            # Преобразование оставшихся строк в список списков
            idx_lines = [line.decode('utf-8').strip().split(';') for line in index_f.readlines()]
            # Пересортировка по VIN  
            sorted_idx_lines = sorted(idx_lines, key=lambda x: x[key_position])
            new_lines = [(';'.join(line).ljust(self.FIXED_LINE_SIZE-1)+'\n') for line in sorted_idx_lines]
                 
            #print(f'Длина новых строк - {len(new_lines[0])},{len(new_lines[1])}, {len(new_lines[2])},...,{len(new_lines[-1])}')
            cars_header_for_write = (';'.join(cars_index_header).ljust(self.FIXED_LINE_SIZE-1)+'\n')

            # Запись в новый файл
            new_index_f.write(cars_header_for_write)
            for one_new_line in new_lines:
                new_index_f.write(one_new_line)

        # Обновление файла
        sorted_file = f'{self.root_directory_path}sorted_cars_index.txt'
        old_file = f'{self.root_directory_path}cars_index.txt'
        os.replace(sorted_file, old_file)

        #print(f'status = {required_car[status_position]}')
        # Обновление файлов продаж если, автомобиль имеет статус 'sold'
        sales_vin_index = False
        sales_line = False
        if required_car[status_position] == 'sold':
            # Обращение к индексу продаж:
            with open(f'{self.root_directory_path}sales_index.txt', mode='r') as sales_index_f,\
                 open(f'{self.root_directory_path}temp_sales_index.txt', mode='w', newline='') as temp_sales_index_f:
                # Динамическое обращение к заголовку
                sales_index_header = sales_index_f.readline().strip().split(';')
                sales_key_pos = sales_index_header.index('key')
                sales_index_pos = sales_index_header.index('index')
                # Поиск старого VIN в ключе индекса
                sales_index_lines = [line.strip().split(';') for line in sales_index_f.readlines()]
                for line in sales_index_lines:
                    # Исправление VIN 
                    if line[sales_key_pos] == vin:
                        # Сохранение индекса (номера строки в основном файле)
                        sales_vin_index = int(line[sales_index_pos])
                        # Замена VIN
                        line[sales_key_pos] = new_vin
                # Сортировка с учётом нового значения
                sorted_sales_index_lines = sorted(sales_index_lines, key=lambda x: x[sales_key_pos])
                # Запись строки заголовка
                temp_sales_index_f.write( ';'.join(sales_index_header).ljust(self.FIXED_LINE_SIZE-1)+'\n' )
                # Запись строк
                for line in sorted_sales_index_lines:
                    line_to_write = ';'.join(line).ljust(self.FIXED_LINE_SIZE-1)+'\n'
                    temp_sales_index_f.write(line_to_write)

            # Замена файла новым 
            new_sales_index_f = f'{self.root_directory_path}temp_sales_index.txt'
            ols_sales_index_f = f'{self.root_directory_path}sales_index.txt'

            os.replace(new_sales_index_f, ols_sales_index_f)

        # Обращение к файлу продаж
        if sales_vin_index:
            with open(f'{self.root_directory_path}sales.txt', mode='r+b') as sales_f:
                sales_header = sales_f.readline().decode('utf-8').strip().split(';')
                sales_vin_pos = sales_header.index('car_vin')
                # Перемещение строки по индексу
                sales_f.seek(sales_vin_index*self.FIXED_LINE_SIZE)
                # Сохранение строки
                sales_line = sales_f.readline().decode('utf-8').strip().split(';')
                # Замена VIN 
                sales_line[sales_vin_pos] = new_vin
                # Реконструкция строки
                edited_sales_line = (';'.join(sales_line).ljust(self.FIXED_LINE_SIZE-1) + '\n').encode('utf-8')
                # Возвращение к началу строки
                sales_f.seek(sales_vin_index*self.FIXED_LINE_SIZE)
                sales_f.write(edited_sales_line)
                 
        if sales_line:                                
            print(f'Номер строки в продажах для vin - {sales_vin_index}, строка - {';'.join(sales_line).encode('utf-8')}')
        
        res_object = Car(
            vin = required_car[vin_position],
            model = required_car[model_position],
            price = required_car[price_position],
            date_start = datetime.strptime(required_car[date_position], '%Y-%m-%d'),
            status = CarStatus(required_car[status_position])
        )

        return res_object

In [25]:
CarService('data/').update_vin('3WVBW2M56ID967749', '3WVBW2M56ID967749')

Номер строки в продажах для vin - 124922, строка - b'119#892;3WVBW2M56ID967749;3547.63;2023-07-12'


Car(vin='3WVBW2M56ID967749', model=115, price=Decimal('2856.63'), date_start=datetime.datetime(2024, 2, 16, 0, 0), status=<CarStatus.sold: 'sold'>)

###### 6. Удаление продажи:

In [26]:
class CarService:
    def __init__(self, root_directory_path: str) -> None:
        self.root_directory_path = root_directory_path
        self.FIXED_LINE_SIZE = 200

    def revert_sale(self, sales_number: str) -> Car:
        # Обращение к файлу продаж
        with open(f'{self.root_directory_path}sales.txt', 'r+', newline='') as sales_f:
            # Обработка заголовка для динамического обращения к элементам строки
            sales_header = sales_f.readline().strip().split(';')
            sales_number_pos = sales_header.index('sales_number')
            sold_car_vin_pos = sales_header.index('car_vin')
            # Построчное чтение файла
            sold_car_line = None
            sold_car_vin = None
            line_pos = 0
            for line in sales_f:
                line = line.strip().split(';')
                line_pos += 1
                if line[sales_number_pos] == sales_number:
                    sold_car_line = line
                    sold_car_vin = line[sold_car_vin_pos]
                    break # Прерывание цикла после нахождения нужной строки

            if sold_car_line == None:
                raise ValueError('Продажи с введенным номером не существует; Проверьте правильность значения.')
                
            sales_f.seek(line_pos*self.FIXED_LINE_SIZE)
            edited_line = sold_car_line
            edited_line[sales_number_pos] = 'deleted'
            #line_in_pos = sales_f.readline().strip().split(';')
            sales_f.write(';'.join(edited_line).ljust(self.FIXED_LINE_SIZE-1)+'\n')

        # Обращение к файлу с индексом автомобилей
        with open(f'{self.root_directory_path}cars_index.txt', 'r+', encoding='utf-8', newline='') as cars_index_f:
            # Обработка заголовка
            cars_index_header = cars_index_f.readline().strip().split(';')
            cars_index_pos = cars_index_header.index('index')
            cars_key_pos = cars_index_header.index('key')

            # Поиск индекса проданного авто
            sold_car_cars_index = None
            for line in cars_index_f:
                line = line.strip().split(';')
                if line[cars_key_pos] == sold_car_vin:
                    sold_car_cars_index = line[cars_index_pos]

            if sold_car_cars_index == None:
                print(f'Проданный автомобиль - {sold_car_vin}')
                raise ValueError('Проданный автомобиль не найден в базе VIN-номеров; Продажа была удалена.')
                
        # Обращение к файлу с базой автомобилей
        with open(f'{self.root_directory_path}cars.txt', 'r+', encoding='utf-8', newline='') as cars_f:
            # Обработка заголовка
            cars_header = cars_f.readline().strip().split(';')
            cars_vin_pos = cars_header.index('vin')
            cars_status_pos = cars_header.index('status')
            cars_model_pos = cars_header.index('model')
            cars_price_pos = cars_header.index('price')
            cars_date_start_pos = cars_header.index('date_start')

            # Переход на нужную строку
            cars_f.seek(self.FIXED_LINE_SIZE*int(sold_car_cars_index))
            sold_car_cars_line = cars_f.readline().strip().split(';')
            # Проверка на случай некорректно выбранной строки
            if sold_car_cars_line[cars_vin_pos]!=sold_car_vin:
                raise ValueError('Ошибка в исправлении статуса автомобиля')
            else:
                # Обновление статуса
                sold_car_cars_line[cars_status_pos] = 'available'
            # Возвращение к началу нужной строки
            cars_f.seek(self.FIXED_LINE_SIZE*int(sold_car_cars_index))
            cars_f.write(';'.join(sold_car_cars_line).ljust(self.FIXED_LINE_SIZE-1)+'\n')
            
        # Возвращение обновлённых данных об автомобиле
        result_car = Car(
            vin = sold_car_cars_line[cars_vin_pos],
            model = int(sold_car_cars_line[cars_model_pos]),
            price = decimal.Decimal(sold_car_cars_line[cars_price_pos]),
            date_start = datetime.strptime(sold_car_cars_line[cars_date_start_pos], '%Y-%m-%d'),
            status = sold_car_cars_line[cars_status_pos]
        )
        
        return result_car


In [27]:
CarService('data/').revert_sale('15#65') #4OFTI5O96PX644978

Car(vin='6AFRP3P05HK817446', model=43, price=Decimal('2147.55'), date_start=datetime.datetime(2024, 5, 21, 0, 0), status=<CarStatus.available: 'available'>)

###### 7. ТОП-3 продаваемых моделей:

In [28]:
class CarService:
    def __init__(self, root_directory_path: str) -> None:
        self.root_directory_path = root_directory_path
        self.FIXED_LINE_SIZE = 200
        
    def top_models_by_sales(self) -> list[ModelSaleStats]:
        sold_models_count = {}
        # Обращение к файлу Cars.txt
        # Файл sales.txt менее приоритетен ввиду наличия удалённых продаж (sales_number = 'deleted')
        with open(f'{self.root_directory_path}cars.txt', mode='r', encoding='utf-8') as cars_f:
            cars_header = cars_f.readline().strip().split(';')
            cars_model_index = cars_header.index('model')
            cars_status_index = cars_header.index('status')
    
            # Построчная обработка автомобилей
            for line in cars_f:
                line = line.strip().split(';')
                line_status = line[cars_status_index]
                line_model = int(line[cars_model_index])
                # Остановка на строках с проданными авто
                if line_status == 'sold':
                    # Запись id modeli с счётчиком количества вхождения
                    if line_model in sold_models_count:
                        sold_models_count[line_model] += 1
                    else:
                        sold_models_count[line_model] = 1
                        
        # Сортировка словарей по ключу, выбор первых трёх
        sorted_models_count = sorted(sold_models_count.items(), key = lambda x: x[1], reverse=True)
        top_3_sold_models = dict(sorted_models_count[:3]) 
        # Поиск информации о названии и бренде моделей
        with open(f'{self.root_directory_path}models.txt', mode='r', encoding='utf-8') as models_f:
            # Обработка заголовка
            models_header = models_f.readline().strip().split(';')
            models_id_pos = models_header.index('id')
            models_name_pos = models_header.index('name')
            models_brand_pos = models_header.index('brand')
            
            # Поиск и запись атрибутов
            top_3_sold_models_info = {}
            for line in models_f.readlines():
                line = line.strip().split(';')
                line_id = int(line[models_id_pos])
                # Остановка на необходимой строке
                if line_id in top_3_sold_models:
                    top_3_sold_models_info[line_id] = {}
                    top_3_sold_models_info[line_id]['car_model_name'] = line[models_name_pos]
                    top_3_sold_models_info[line_id]['brand'] = line[models_brand_pos]
                    top_3_sold_models_info[line_id]['sales_number'] = top_3_sold_models[line_id]
                # Если все 3 модели найдены
                if len(top_3_sold_models_info) == 3:
                    # Прерывание цикла
                    break

        result_list = []
        # Преобразование внутренних словарей в объекты ModelSaleStats и включение в список
        [result_list.append(
            ModelSaleStats(
                car_model_name = top_3_sold_models_info[sold_model]['car_model_name'],
                brand          = top_3_sold_models_info[sold_model]['brand'],
                sales_number   = top_3_sold_models_info[sold_model]['sales_number']
            )
        )
        for sold_model in top_3_sold_models_info]
        # Сортировка по убыванию
        result_list = sorted(result_list, key=lambda x: x.sales_number, reverse=True)

        return result_list
        

In [29]:
CarService('data/').top_models_by_sales()

[ModelSaleStats(car_model_name='Z-6', brand='T', sales_number=288),
 ModelSaleStats(car_model_name='C-9', brand='S', sales_number=288),
 ModelSaleStats(car_model_name='F-3', brand='O', sales_number=287)]

---

##### Тесты чтения:

In [30]:
data = CarService('data/')

In [31]:
with open(f'{data.root_directory_path}cars_index.txt', mode='r', encoding='utf-8') as f:
    jump_size = 1200
    f.seek(jump_size)
    row = f.readline()
    print(f"Строка {int(jump_size / 200)}: {row.rstrip()}, длина с пробелами - {len(row)}")

Строка 6: 0AAAV6H43MF053835;348655, длина с пробелами - 200


In [32]:
with open(f'{data.root_directory_path}cars.txt', mode='r', encoding='utf-8') as f:
    jump_size = 169001*200
    f.seek(jump_size)
    row = f.readline()
    print(f"Строка {int(jump_size / 200)}: {row.rstrip()}, длина с пробелами - {len(row)}")

Строка 169001: 2GKOG3U16KC590054;184;2755.9;2022-05-09;available, длина с пробелами - 200


---

Проверка на длину в байтах:

In [33]:
with open('data/cars_index.txt', mode='rb') as f:
    # Читаем первые 200 байт (Строка 1)
    chunk1 = f.read(200)
    print(f"Размер первой порции: {len(chunk1)} байт")
    print(f"Строка 1 в байтах: {chunk1[:50]}... (всего {len(chunk1)} байт)")
    
    # Читаем следующие 200 байт (Строка 2)
    chunk2 = f.read(200)
    print(f"Строка 2 в байтах: {chunk2[:50]}... (всего {len(chunk2)} байт)")


Размер первой порции: 200 байт
Строка 1 в байтах: b'key;index                                         '... (всего 200 байт)
Строка 2 в байтах: b'0AAAA6R32VH044942;271142                          '... (всего 200 байт)


---

##### Список заданий:

class CarService:
    def __init__(self, root_directory_path: str) -> None:
        self.root_directory_path = root_directory_path
    
    # Задание 1. Сохранение автомобилей и моделей
    def add_model(self, model: Model):
        pass|
        
    # Задание 1. Сохранение автомобилей и моделей
    def add_car(self, car: Car) -> Car:
        pass

    # Задание 2. Сохранение продаж.
    def sell_car(self, sale: Sale) -> Car:
        pass

    # Задание 3. Доступные к продаже
    def get_cars(self, status: CarStatus) -> list[Car]:
        pass
        
    # Задание 4. Детальная информация
    def get_car_info(self, vin: str) -> CarFullInfo | None:
        pass#raise NotImplementedError

    # Задание 5. Обновление ключевого поля
    def update_vin(self, vin: str, new_vin: str) -> Car:
        pass#raise NotImplementedError

    # Задание 6. Удаление продажи
    def revert_sale(self, sales_number: str) -> Car:
        pass#raise NotImplementedError

    # Задание 7. Самые продаваемые модели
    def top_models_by_sales(self) -> list[ModelSaleStats]:
        pass

---